# AO Robustness: Open-Ended Q/A vs Binary Classification

**Condition A (Baseline)**: LatentQA + Classification + PastLens  
**Condition B (Open-Ended)**: LatentQA + SQuAD QA + PastLens

Does richer open-ended Q/A training data improve oracle quality over binary classification?

## 1. Setup

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# All repo paths are relative to activation_oracles/
_AO_ROOT = os.path.expanduser("~/ao-robustness/activation_oracles")
os.chdir(_AO_ROOT)

import gc
import json
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from peft import LoraConfig

from nl_probes.configs.sft_config import SelfInterpTrainingConfig
from nl_probes.dataset_classes.act_dataset_manager import DatasetLoaderConfig
from nl_probes.dataset_classes.classification import ClassificationDatasetConfig, ClassificationDatasetLoader
from nl_probes.dataset_classes.squad_qa import SquadQADatasetConfig, SquadQADatasetLoader
from nl_probes.sft import _ensure_datasets_exist, build_datasets, mk_cfg
from nl_probes.utils.activation_utils import get_hf_submodule
from nl_probes.utils.common import load_model, load_tokenizer
from nl_probes.utils.eval import run_evaluation, score_eval_responses

from notebooks.train_condition import (
    CLASSIFICATION_DATASETS,
    build_condition_loaders,
)

/nfs/nhome/live/jbauer/ao-robustness/activation_oracles/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
MODEL_NAME = "Qwen/Qwen3-8B"
MODEL_NAME_STR = MODEL_NAME.split("/")[-1].replace(".", "_")
LAYER_PERCENTS = [25, 50, 75]
BATCH_SIZE = 16
DTYPE = torch.bfloat16
DEVICE = torch.device("cuda")
STEERING_COEFFICIENT = 1.0
HOOK_LAYER = 1
GENERATION_KWARGS = {"do_sample": False, "temperature": 0.0, "max_new_tokens": 20}

CONDITION_A_SUFFIX = f"_baseline_cls_{MODEL_NAME_STR}"
CONDITION_B_SUFFIX = f"_openended_squad_{MODEL_NAME_STR}"

CONDITION_A_LORA = "jbauer/ao-condition-a-qwen3-8b"
CONDITION_B_LORA = "jbauer/ao-condition-b-qwen3-8b"

## 2. Build Loaders

In [ ]:
save_acts = False
model_kwargs = {}

loaders_a = build_condition_loaders("a", MODEL_NAME, LAYER_PERCENTS, BATCH_SIZE, save_acts, model_kwargs)
loaders_b = build_condition_loaders("b", MODEL_NAME, LAYER_PERCENTS, BATCH_SIZE, save_acts, model_kwargs)

print(f"Condition A loaders: {len(loaders_a)}")
for l in loaders_a:
    print(f"  {l.dataset_config.dataset_name}: train={l.dataset_config.num_train}, test={l.dataset_config.num_test}")

print(f"\nCondition B loaders: {len(loaders_b)}")
for l in loaders_b:
    print(f"  {l.dataset_config.dataset_name}: train={l.dataset_config.num_train}, test={l.dataset_config.num_test}")

Condition A loaders: 23
  latentqa: train=100000, test=0
  classification_geometry_of_truth: train=6000, test=250
  classification_geometry_of_truth: train=6000, test=250
  classification_relations: train=6000, test=250
  classification_relations: train=6000, test=250
  classification_sst2: train=6000, test=250
  classification_sst2: train=6000, test=250
  classification_md_gender: train=6000, test=250
  classification_md_gender: train=6000, test=250
  classification_snli: train=6000, test=250
  classification_snli: train=6000, test=250
  classification_ag_news: train=0, test=250
  classification_ag_news: train=0, test=250
  classification_ner: train=6000, test=250
  classification_ner: train=6000, test=250
  classification_tense: train=6000, test=250
  classification_tense: train=6000, test=250
  classification_language_identification: train=6000, test=250
  classification_language_identification: train=6000, test=250
  classification_singular_plural: train=0, test=250
  classificatio

## 3. Inspect Data

In [ ]:
# Materialize datasets (rank 0 only, no DDP needed for inspection)
_ensure_datasets_exist(loaders_a)
_ensure_datasets_exist(loaders_b)

Loaded 64840 datapoints from sft_training_data/latentqa_model_Qwen3-8B_n_100000_save_acts_False_train_4f839d642efb.pt
Loaded 36000 datapoints from sft_training_data/classification_geometry_of_truth_model_Qwen3-8B_n_6000_save_acts_False_train_0261fc2c2e71.pt
Loaded 1500 datapoints from sft_training_data/classification_geometry_of_truth_model_Qwen3-8B_n_250_save_acts_False_test_c0cbb446c24d.pt
Loaded 18000 datapoints from sft_training_data/classification_geometry_of_truth_model_Qwen3-8B_n_6000_save_acts_False_train_35bc8469c30e.pt
Loaded 750 datapoints from sft_training_data/classification_geometry_of_truth_model_Qwen3-8B_n_250_save_acts_False_test_17e2c609c12c.pt
Loaded 36000 datapoints from sft_training_data/classification_relations_model_Qwen3-8B_n_6000_save_acts_False_train_d5fbc9857426.pt
Loaded 1500 datapoints from sft_training_data/classification_relations_model_Qwen3-8B_n_250_save_acts_False_test_eac573d2fa9e.pt
Loaded 18000 datapoints from sft_training_data/classification_relati

In [ ]:
cfg_inspect = SelfInterpTrainingConfig(model_name=MODEL_NAME, layer_percents=LAYER_PERCENTS, train_batch_size=BATCH_SIZE)
cfg_inspect.finalize(dataset_loaders=loaders_a)

train_a, eval_a = build_datasets(cfg_inspect, loaders_a, max_len_percentile=None, window_mult=None)
print(f"Condition A: {len(train_a)} train, eval keys: {list(eval_a.keys())}")

cfg_inspect_b = SelfInterpTrainingConfig(model_name=MODEL_NAME, layer_percents=LAYER_PERCENTS, train_batch_size=BATCH_SIZE)
cfg_inspect_b.finalize(dataset_loaders=loaders_b)

train_b, eval_b = build_datasets(cfg_inspect_b, loaders_b, max_len_percentile=None, window_mult=None)
print(f"Condition B: {len(train_b)} train, eval keys: {list(eval_b.keys())}")

Loaded 64840 datapoints from sft_training_data/latentqa_model_Qwen3-8B_n_100000_save_acts_False_train_4f839d642efb.pt
Loaded 36000 datapoints from sft_training_data/classification_geometry_of_truth_model_Qwen3-8B_n_6000_save_acts_False_train_0261fc2c2e71.pt
Loaded 1500 datapoints from sft_training_data/classification_geometry_of_truth_model_Qwen3-8B_n_250_save_acts_False_test_c0cbb446c24d.pt
Loaded 18000 datapoints from sft_training_data/classification_geometry_of_truth_model_Qwen3-8B_n_6000_save_acts_False_train_35bc8469c30e.pt
Loaded 750 datapoints from sft_training_data/classification_geometry_of_truth_model_Qwen3-8B_n_250_save_acts_False_test_17e2c609c12c.pt
Loaded 36000 datapoints from sft_training_data/classification_relations_model_Qwen3-8B_n_6000_save_acts_False_train_d5fbc9857426.pt
Loaded 1500 datapoints from sft_training_data/classification_relations_model_Qwen3-8B_n_250_save_acts_False_test_eac573d2fa9e.pt
Loaded 18000 datapoints from sft_training_data/classification_relati

In [ ]:
# Datapoint type distributions
for label, data in [("Condition A", train_a), ("Condition B", train_b)]:
    counts = Counter(dp.datapoint_type for dp in data)
    print(f"\n{label} datapoint types:")
    for k, v in sorted(counts.items(), key=lambda x: -x[1]):
        print(f"  {k}: {v}")


Condition A datapoint types:
  past_lens: 584488
  classification_tense: 54000
  classification_geometry_of_truth: 54000
  classification_md_gender: 54000
  classification_ner: 54000
  classification_snli: 54000
  classification_sst2: 54000
  classification_relations: 54000
  latentqa_control: 33464
  latentqa_stimulus_completion: 15688
  latentqa_stimulus: 15688

Condition B datapoint types:
  past_lens: 584488
  squad_qa: 360000
  latentqa_control: 33464
  latentqa_stimulus_completion: 15688
  latentqa_stimulus: 15688


In [ ]:
# Decode a few sample prompts from each condition
tokenizer = load_tokenizer(MODEL_NAME)

for label, data in [("Condition A", train_a), ("Condition B", train_b)]:
    print(f"\n{'='*60}\n{label} samples\n{'='*60}")
    seen_types = set()
    for dp in data:
        if dp.datapoint_type in seen_types:
            continue
        seen_types.add(dp.datapoint_type)
        prompt_text = tokenizer.decode(dp.input_ids, skip_special_tokens=False)
        print(f"\n--- {dp.datapoint_type} (layer={dp.layer}) ---")
        print(f"Target: {dp.target_output}")
        print(f"Prompt (truncated): {prompt_text[:300]}...")
        if len(seen_types) >= 6:
            break

# Free memory
del train_a, train_b, eval_a, eval_b
gc.collect()

📦 Loading tokenizer...

Condition A samples

--- classification_tense (layer=27) ---
Target: No
Prompt (truncated): <|im_start|>user
Layer: 27
 ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? 
Answer with 'Yes' or 'No' only. Can we classify this sentence as being in the future tense?<|im_end|>
<|im_start|>assistant
<think>

</think>

No<|im_end|>
...

--- classification_geometry_of_truth (layer=18) ---
Target: Yes
Prompt (truncated): <|im_start|>user
Layer: 18
 ? ? ? ? ? ? ? ? ? ? ? ? 
Answer with 'Yes' or 'No' only. Do you think this checks out?<|im_end|>
<|im_start|>assistant
<think>

</think>

Yes<|im_end|>
...

--- past_lens (layer=9) ---
Target:  thấy qua bầu
Prompt (truncated): <|im_start|>user
Layer: 9
 ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? 
Can you predict the next 3 tokens that come after this?<|im_end|>
<|im_start|>assistant
<think>

</think>

 thấy qua bầu<|im_end|>
...

--- latentqa_control (layer=9) ---
Target: The assistant engages the user through thought-provoking paradoxes, 

18

## 4. Train

Run these in a terminal (from the `activation_oracles/` directory):

In [ ]:
NGPU = 1  # adjust to your setup

print("# Condition A (baseline: LatentQA + Classification + PastLens)")
print(f"torchrun --nproc_per_node={NGPU} notebooks/train_condition.py --condition a --model_name {MODEL_NAME}")
print()
print("# Condition B (open-ended: LatentQA + SQuAD + PastLens)")
print(f"torchrun --nproc_per_node={NGPU} notebooks/train_condition.py --condition b --model_name {MODEL_NAME}")
print()
print("# With srun (SLURM):")
print(f"srun --gpus={NGPU} torchrun --nproc_per_node={NGPU} notebooks/train_condition.py --condition a --model_name {MODEL_NAME}")
print(f"srun --gpus={NGPU} torchrun --nproc_per_node={NGPU} notebooks/train_condition.py --condition b --model_name {MODEL_NAME}")

# Condition A (baseline: LatentQA + Classification + PastLens)
torchrun --nproc_per_node=1 notebooks/train_condition.py --condition a --model_name Qwen/Qwen3-8B

# Condition B (open-ended: LatentQA + SQuAD + PastLens)
torchrun --nproc_per_node=1 notebooks/train_condition.py --condition b --model_name Qwen/Qwen3-8B

# With srun (SLURM):
srun --gpus=1 torchrun --nproc_per_node=1 notebooks/train_condition.py --condition a --model_name Qwen/Qwen3-8B
srun --gpus=1 torchrun --nproc_per_node=1 notebooks/train_condition.py --condition b --model_name Qwen/Qwen3-8B


## 5. Evaluate

Load both checkpoints and run classification eval (10 datasets).

In [ ]:
# Build classification test loaders for evaluation (same for both conditions)
EVAL_TEST_SIZE = 250
eval_batch_size = 128

eval_loaders = []
for ds_name in CLASSIFICATION_DATASETS:
    cfg_ds = ClassificationDatasetConfig(
        classification_dataset_name=ds_name,
        max_end_offset=-3, min_end_offset=-3,
        max_window_size=1, min_window_size=1,
    )
    eval_loaders.append(ClassificationDatasetLoader(
        dataset_config=DatasetLoaderConfig(
            custom_dataset_params=cfg_ds,
            num_train=0, num_test=EVAL_TEST_SIZE,
            splits=["test"],
            model_name=MODEL_NAME,
            layer_percents=LAYER_PERCENTS,
            save_acts=True,
            batch_size=eval_batch_size,
        ),
    ))

# Also create SQuAD test loader for open-ended eval
squad_eval_loader = SquadQADatasetLoader(
    dataset_config=DatasetLoaderConfig(
        custom_dataset_params=SquadQADatasetConfig(
            max_end_offset=-3, min_end_offset=-3,
            max_window_size=1, min_window_size=1,
        ),
        num_train=0, num_test=EVAL_TEST_SIZE,
        splits=["test"],
        model_name=MODEL_NAME,
        layer_percents=LAYER_PERCENTS,
        save_acts=True,
        batch_size=eval_batch_size,
    ),
)

all_eval_data = {}
for loader in eval_loaders:
    all_eval_data[loader.dataset_config.dataset_name] = loader.load_dataset("test")
all_eval_data["squad_qa"] = squad_eval_loader.load_dataset("test")

print(f"Eval datasets: {list(all_eval_data.keys())}")
for k, v in all_eval_data.items():
    print(f"  {k}: {len(v)} examples")

📦 Loading tokenizer...
🧠 Loading model...


Loading checkpoint shards: 100%|██████████| 5/5 [02:27<00:00, 29.42s/it]


Saved 2250 test datapoints to sft_training_data/classification_geometry_of_truth_model_Qwen3-8B_n_250_save_acts_True_test_7c97022dd6a2.pt
Loaded 2250 datapoints from sft_training_data/classification_geometry_of_truth_model_Qwen3-8B_n_250_save_acts_True_test_7c97022dd6a2.pt
📦 Loading tokenizer...
🧠 Loading model...


Loading checkpoint shards: 100%|██████████| 5/5 [00:06<00:00,  1.34s/it]


Saved 2250 test datapoints to sft_training_data/classification_relations_model_Qwen3-8B_n_250_save_acts_True_test_084ee9f1c7eb.pt
Loaded 2250 datapoints from sft_training_data/classification_relations_model_Qwen3-8B_n_250_save_acts_True_test_084ee9f1c7eb.pt
📦 Loading tokenizer...
🧠 Loading model...


...

Saved 2250 test datapoints to sft_training_data/classification_tense_model_Qwen3-8B_n_250_save_acts_True_test_24621dd81a04.pt
Loaded 2250 datapoints from sft_training_data/classification_tense_model_Qwen3-8B_n_250_save_acts_True_test_24621dd81a04.pt
📦 Loading tokenizer...
🧠 Loading model...


Loading checkpoint shards: 100%|██████████| 5/5 [00:06<00:00,  1.32s/it]

Unexpected error during forward pass: CUDA out of memory. Tried to allocate 2.13 GiB. GPU 0 has a total capacity of 23.52 GiB of which 967.38 MiB is free. Including non-PyTorch memory, this process has 22.37 GiB memory in use. Of the allocated memory 22.07 GiB is allocated by PyTorch, and 60.47 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.13 GiB. GPU 0 has a total capacity of 23.52 GiB of which 967.38 MiB is free. Including non-PyTorch memory, this process has 22.37 GiB memory in use. Of the allocated memory 22.07 GiB is allocated by PyTorch, and 60.47 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

: 

In [ ]:
# Load model
tokenizer = load_tokenizer(MODEL_NAME)
model = load_model(MODEL_NAME, DTYPE)
model.eval()
submodule = get_hf_submodule(model, HOOK_LAYER)

# Add dummy adapter for PeftModel API
dummy_config = LoraConfig()
model.add_adapter(dummy_config, adapter_name="default")

In [ ]:
# Run evaluation for each condition + base model
lora_configs = {
    "Base Model": None,
    "Condition A (Classification)": CONDITION_A_LORA,
    "Condition B (SQuAD)": CONDITION_B_LORA,
}

all_results = {}  # results[ds_name][condition_label] = (format_correct, ans_correct)

for condition_label, lora_path in lora_configs.items():
    print(f"\n{'='*60}\nEvaluating: {condition_label} (lora={lora_path})\n{'='*60}")
    for ds_name, eval_data in all_eval_data.items():
        results = run_evaluation(
            eval_data=eval_data,
            model=model,
            tokenizer=tokenizer,
            submodule=submodule,
            device=DEVICE,
            dtype=DTYPE,
            global_step=-1,
            lora_path=lora_path,
            eval_batch_size=eval_batch_size,
            steering_coefficient=STEERING_COEFFICIENT,
            generation_kwargs=GENERATION_KWARGS,
        )

        # Score classification datasets with yes/no valid answers
        if ds_name.startswith("classification_"):
            fmt, acc = score_eval_responses(results, eval_data, valid_answers=["yes", "no"])
        else:
            # For SQuAD, format_correct doesn't apply; just compute exact match
            fmt, acc = score_eval_responses(results, eval_data, valid_answers=[])

        all_results.setdefault(ds_name, {})[condition_label] = {"format_correct": fmt, "ans_correct": acc}
        print(f"  {ds_name}: format={fmt:.3f}, accuracy={acc:.3f}")

In [ ]:
# Clean up model
del model, submodule
torch.cuda.empty_cache()
gc.collect()

## 6. Analysis

In [ ]:
# Build comparison dataframe
rows = []
for ds_name, cond_results in all_results.items():
    for cond_label, metrics in cond_results.items():
        rows.append({"dataset": ds_name, "condition": cond_label, "accuracy": metrics["ans_correct"], "format_correct": metrics["format_correct"]})

df = pd.DataFrame(rows)
print(df.pivot(index="dataset", columns="condition", values="accuracy").to_string())

In [ ]:
# Bar chart: accuracy by dataset for each condition
pivot = df.pivot(index="dataset", columns="condition", values="accuracy")

fig, ax = plt.subplots(figsize=(14, 6))
pivot.plot(kind="bar", ax=ax)
ax.set_ylabel("Answer Accuracy")
ax.set_title(f"AO Robustness: Classification Accuracy by Condition ({MODEL_NAME})")
ax.set_ylim(0, 1.0)
ax.legend(title="Condition")
ax.yaxis.grid(True, linestyle="--", alpha=0.4)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Delta table: Condition B - Condition A
cond_a_label = "Condition A (Classification)"
cond_b_label = "Condition B (SQuAD)"

if cond_a_label in pivot.columns and cond_b_label in pivot.columns:
    delta = pivot[cond_b_label] - pivot[cond_a_label]
    delta_df = pd.DataFrame({"B - A delta": delta, "Condition A": pivot[cond_a_label], "Condition B": pivot[cond_b_label]})
    print("Per-dataset accuracy delta (Condition B - Condition A):")
    print(delta_df.to_string())
    print(f"\nMean delta: {delta.mean():.4f}")
    print(f"Datasets where B > A: {(delta > 0).sum()} / {len(delta)}")
    print(f"Datasets where A > B: {(delta < 0).sum()} / {len(delta)}")

In [ ]:
# Delta bar chart
if cond_a_label in pivot.columns and cond_b_label in pivot.columns:
    colors = ["green" if d > 0 else "red" for d in delta]
    fig, ax = plt.subplots(figsize=(14, 5))
    delta.plot(kind="bar", ax=ax, color=colors)
    ax.set_ylabel("Accuracy Delta (B - A)")
    ax.set_title(f"Condition B (SQuAD) vs Condition A (Classification) ({MODEL_NAME})")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.yaxis.grid(True, linestyle="--", alpha=0.4)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()